# Public Transport Delay Duration Prediction\n\nTrain a regression model to predict **delay duration** (minutes) using weather, traffic, schedule, and event features.\n\nThis notebook uses a **RandomForestRegressor** with a preprocessing pipeline (imputation + one-hot encoding).

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Load dataset
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
data_path = os.path.join('data', 'public_transport_delays.csv')
df = pd.read_csv(data_path)
df.head()

In [ ]:
# Pick a delay-duration target column
candidate_targets = [
    'actual_arrival_delay_min',
    'actual_departure_delay_min',
    'delay_minutes'
]
target = next((c for c in candidate_targets if c in df.columns), None)
if target is None:
    raise ValueError(f'No delay duration target found. Expected one of {candidate_targets}')

# Drop rows where target is missing
df = df.dropna(subset=[target])

# Features/labels
X = df.drop(columns=[target])
y = df[target]

X.head()

In [ ]:
# Split columns by type
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.columns.difference(cat_cols)

# Preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

# Pipeline
clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', model)
])

clf

In [ ]:
# Train
clf.fit(X_train, y_train)

In [ ]:
# Evaluate
pred = clf.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred, squared=False)
r2 = r2_score(y_test, pred)

print(f'MAE: {mae:.3f}')
print(f'RMSE: {rmse:.3f}')
print(f'R2: {r2:.3f}')

In [ ]:
# Save model pipeline
os.makedirs('outputs', exist_ok=True)
joblib.dump(clf, 'outputs/delay_duration_model.pkl')
print('Saved to outputs/delay_duration_model.pkl')